In [1]:
from dotenv import load_dotenv
import pyobis
print("OK")

OK


In [2]:
import os

print(os.getcwd())
print(os.path.exists("./escales6.json"))
print(os.path.exists("./data_texts/20000lieues_fr.txt"))
print(os.path.exists("./data_texts/20000lieues_an.txt"))

C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks
True
True
True


In [3]:
import json

with open("./escales6.json", "r", encoding="utf-8") as f:
    escales = json.load(f)

with open("./data_texts/20000lieues_fr.txt", "r", encoding="utf-8") as f:
    roman_fr = f.read()

with open("./data_texts/20000lieues_an.txt", "r", encoding="utf-8") as f:
    roman_en = f.read()

print(type(escales), len(escales))
print(type(roman_fr), len(roman_fr))
print(type(roman_en), len(roman_en))
print(escales[0])

<class 'list'> 32
<class 'str'> 908396
<class 'str'> 617239
{'Escales du Nautilus': {'fr': 'Île de Crespo', 'en': 'Crespo Island'}, 'Date': {'fr': '1er Décembre 1866', 'en': 'December 1st, 1866'}, 'Latitude_Décimal': 33.18497, 'Longitude_Décimal': 169.85446, 'Océan/Mer': {'fr': 'Océan Pacifique Sud', 'en': 'South Pacific Ocean'}, 'Event': {'fr': "Début de l'aventure - Première chasse sous-marine - Une forêt sous-marine - Une monstrueuse araignée de mer.", 'en': 'Beginning of the adventure - First underwater hunt - A submarine forest - A monstrous sea spider.'}}


In [4]:
stop = escales[0]

print(stop["Escales du Nautilus"]["fr"])
print(stop["Escales du Nautilus"]["en"])
print(stop["Event"]["fr"])
print(stop["Event"]["en"])

motif_fr = "Crespo"
motif_en = "Crespo"

print("FR index:", roman_fr.find(motif_fr))
print("EN index:", roman_en.find(motif_en))

Île de Crespo
Crespo Island
Début de l'aventure - Première chasse sous-marine - Une forêt sous-marine - Une monstrueuse araignée de mer.
Beginning of the adventure - First underwater hunt - A submarine forest - A monstrous sea spider.
FR index: 224907
EN index: 158021


In [5]:
def extraire_contexte(texte, index, taille=1200):
    if index == -1:
        return None
    debut = max(0, index - taille)
    fin = min(len(texte), index + taille)
    return texte[debut:fin]

passage_fr = extraire_contexte(roman_fr, 224907, 1200)
passage_en = extraire_contexte(roman_en, 158021, 1200)

print(passage_fr[:1500])
print("\n" + "="*80 + "\n")
print(passage_en[:1500])

[Illustration: La mer s'enflamma à son regard. (Page 111.)]

«Nautron respoc lorni virch.»

Ce qu'elle signifiait, je ne saurais le dire.

Ces mots prononcés, le second redescendit. Je pensai que le _Nautilus_
allait reprendre sa navigation sous-marine. Je regagnai donc le
panneau, et par les coursives je revins à ma chambre.

Cinq jours s'écoulèrent ainsi, sans que la situation se modifiât.
Chaque matin, je montais sur la plate-forme. La même phrase était
prononcée par le même individu. Le capitaine Nemo ne paraissait pas.

J'avais pris mon parti de ne plus le voir, quand, le 16 novembre,
rentré dans ma chambre avec Ned et Conseil, je trouvai sur la table un
billet à mon adresse.

[Illustration: Je fis honneur au repas. (Page 115.)]

Je l'ouvris d'une main impatiente. Il était écrit d'une écriture
franche et nette, mais un peu gothique et qui rappelait les types
allemands.

Ce billet était libellé en ces termes:

    «Monsieur le professeur Aronnax, à bord du _Nautilus_.

            

In [7]:
def extraire_contexte(texte, mot_cle, taille=900):
    idx = texte.lower().find(mot_cle.lower())
    if idx == -1:
        return None, -1
    debut = max(0, idx - taille)
    fin = min(len(texte), idx + len(mot_cle) + taille)
    return texte[debut:fin], idx

In [8]:
stop = escales[0]
mot_cle = "Crespo"

passage, idx = extraire_contexte(roman_fr, mot_cle)

print(idx)
print(passage[:1500])

224907
ves je revins à ma chambre.

Cinq jours s'écoulèrent ainsi, sans que la situation se modifiât.
Chaque matin, je montais sur la plate-forme. La même phrase était
prononcée par le même individu. Le capitaine Nemo ne paraissait pas.

J'avais pris mon parti de ne plus le voir, quand, le 16 novembre,
rentré dans ma chambre avec Ned et Conseil, je trouvai sur la table un
billet à mon adresse.

[Illustration: Je fis honneur au repas. (Page 115.)]

Je l'ouvris d'une main impatiente. Il était écrit d'une écriture
franche et nette, mais un peu gothique et qui rappelait les types
allemands.

Ce billet était libellé en ces termes:

    «Monsieur le professeur Aronnax, à bord du _Nautilus_.

                                                «16 novembre 1867.

    »Le capitaine Nemo invite monsieur le professeur Aronnax à une
    partie de chasse qui aura lieu demain matin dans ses forêts
    de l'île Crespo. Il espère que rien n'empêchera monsieur le
    professeur d'y assister, et il verra a

In [9]:
def trouver_passage_par_escale(stop, roman_fr, roman_en, taille=900):
    mot_fr = stop["Escales du Nautilus"]["fr"].replace("Île de ", "").replace("Île d'", "").split()[0]
    mot_en = stop["Escales du Nautilus"]["en"].replace("Island", "").split()[0]

    passage_fr, idx_fr = extraire_contexte(roman_fr, mot_fr, taille=taille)
    passage_en, idx_en = extraire_contexte(roman_en, mot_en, taille=taille)

    return {
        "escale_fr": stop["Escales du Nautilus"]["fr"],
        "escale_en": stop["Escales du Nautilus"]["en"],
        "mot_fr": mot_fr,
        "mot_en": mot_en,
        "idx_fr": idx_fr,
        "idx_en": idx_en,
        "passage_fr": passage_fr,
        "passage_en": passage_en,
    }

In [10]:
resultats = []

for stop in escales[:5]:
    resultat = trouver_passage_par_escale(stop, roman_fr, roman_en)
    resultats.append(resultat)

for r in resultats:
    print("=" * 80)
    print(r["escale_fr"], "|", r["idx_fr"], "|", r["mot_fr"])
    print(r["passage_fr"][:500] if r["passage_fr"] else "Aucun passage trouvé")
    print()

Île de Crespo | 224907 | Crespo
ves je revins à ma chambre.

Cinq jours s'écoulèrent ainsi, sans que la situation se modifiât.
Chaque matin, je montais sur la plate-forme. La même phrase était
prononcée par le même individu. Le capitaine Nemo ne paraissait pas.

J'avais pris mon parti de ne plus le voir, quand, le 16 novembre,
rentré dans ma chambre avec Ned et Conseil, je trouvai sur la table un
billet à mon adresse.

[Illustration: Je fis honneur au repas. (Page 115.)]

Je l'ouvris d'une main impatiente. Il était écrit d'une

Archipel des Pomotou | 8453 | Archipel
rme avoir vu, étant à bord du
_Castillan_, en 1857, cet énorme serpent qui n'avait jamais fréquenté
jusqu'alors que les mers de l'ancien _Constitutionnel_.

Alors éclata l'interminable polémique des crédules et des incrédules
dans les sociétés savantes et les journaux scientifiques. La «question
du monstre» enflamma les esprits. Les journalistes, qui font profession
de science en lutte avec ceux qui font profession d'esprit

In [13]:
import re

def nettoyer_texte(texte):
    texte = texte.replace("\ufeff", "")
    texte = re.sub(r"_(.*?)_", r"\1", texte)
    texte = re.sub(r"The Project Gutenberg eBook.*?\n", "", texte, flags=re.IGNORECASE)
    texte = re.sub(r"\[Illustration:.*?\]", "", texte, flags=re.DOTALL)
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte

def decouper_en_phrases(texte):
    return re.split(r'(?<=[\.\!\?])\s+', texte)

def extraire_phrase_contexte(texte, mot_cle, nb_phrases=2):
    texte = nettoyer_texte(texte)
    phrases = decouper_en_phrases(texte)

    for i, phrase in enumerate(phrases):
        if mot_cle.lower() in phrase.lower():
            debut = max(0, i - nb_phrases)
            fin = min(len(phrases), i + nb_phrases + 1)
            return " ".join(phrases[debut:fin]), i
    return None, -1

In [14]:
passage_fr, idx_fr = extraire_phrase_contexte(roman_fr, "Crespo", nb_phrases=2)
print(idx_fr)
print(passage_fr)

2232
Ce billet était libellé en ces termes: «Monsieur le professeur Aronnax, à bord du Nautilus. «16 novembre 1867. »Le capitaine Nemo invite monsieur le professeur Aronnax à une partie de chasse qui aura lieu demain matin dans ses forêts de l'île Crespo. Il espère que rien n'empêchera monsieur le professeur d'y assister, et il verra avec plaisir que ses compagnons se joignent à lui. «Le commandant du Nautilus, «Capitaine NEMO.» «Une chasse!


In [15]:
def enrichir_escale(stop, roman_fr, roman_en):
    mot_fr = "Crespo" if "Crespo" in stop["Escales du Nautilus"]["fr"] else stop["Escales du Nautilus"]["fr"].split()[-1]
    mot_en = "Crespo" if "Crespo" in stop["Escales du Nautilus"]["en"] else stop["Escales du Nautilus"]["en"].split()[0]

    passage_fr, _ = extraire_phrase_contexte(roman_fr, mot_fr, nb_phrases=2)
    passage_en, _ = extraire_phrase_contexte(roman_en, mot_en, nb_phrases=2)

    stop_enrichi = stop.copy()
    stop_enrichi["passage_roman"] = {
        "fr": passage_fr,
        "en": passage_en
    }
    return stop_enrichi

In [16]:
escales_enrichies = []

for stop in escales:
    escales_enrichies.append(enrichir_escale(stop, roman_fr, roman_en))

In [18]:
import json

with open("escales_enrichies.json", "w", encoding="utf-8") as f:
    json.dump(escales_enrichies, f, ensure_ascii=False, indent=2)

In [19]:
import json

with open("escales_enrichies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[0]["Escales du Nautilus"]["fr"])
print(data[0].keys())
print(data[0]["passage_roman"]["fr"][:400])

Île de Crespo
dict_keys(['Escales du Nautilus', 'Date', 'Latitude_Décimal', 'Longitude_Décimal', 'Océan/Mer', 'Event', 'passage_roman'])
Ce billet était libellé en ces termes: «Monsieur le professeur Aronnax, à bord du Nautilus. «16 novembre 1867. »Le capitaine Nemo invite monsieur le professeur Aronnax à une partie de chasse qui aura lieu demain matin dans ses forêts de l'île Crespo. Il espère que rien n'empêchera monsieur le professeur d'y assister, et il verra avec plaisir que ses compagnons se joignent à lui. «Le commandant du 
